# 01 · Fire counts: what the satellites saw

**Project:** Paddy Burnt-Area Intelligence (Satellite Mission Intelligence for India)
**Question for this notebook:** how many VIIRS fire detections did Punjab and Haryana have each paddy season since 2012, *where* and *when in the day* were they seen, and which districts should we study?

**What it produces**

| Output | Where |
|---|---|
| Clean detections (points, with district) | `data/processed/firms_viirs_punjab_haryana_points.parquet` |
| Detections by season and state | `data/processed/firms_detections_by_season_state.csv` |
| Detections by district and season | `data/processed/firms_detections_by_district_season.csv` |
| S-NPP to NOAA-20 ratio (for Notebook 04) | `data/processed/firms_snpp_noaa20_ratio.csv` |
| District shortlist for Checkpoint 1 | `data/processed/district_shortlist.csv` |
| Charts and maps | `maps/notebook01/*.png` |
| Source register rows | `sources/source_register.csv` |

**How to run on your computer (VS Code)**
1. Clone the repository, open the folder in VS Code, and install the libraries into a virtual environment with `pip install -r requirements.txt`. The README has the exact commands.
2. Create a file named `.env` in the project folder, next to `README.md`, containing one line: `FIRMS_MAP_KEY=your_key`. Git ignores this file, so the key is never pushed.
3. Open this notebook, choose your project environment as the kernel (top right; `eo` if you followed the conda steps), then click **Run All**.

**Or run it in Google Colab**
1. Put the project folder in Google Drive at `MyDrive/India-Space-Mission-Intelligence`.
2. Add your FIRMS map key as a Colab secret: key icon in the left sidebar, **Add new secret**, name `FIRMS_MAP_KEY`, paste the key, switch on notebook access.
3. **Runtime → Run all**, and allow Google Drive access when asked.

No GPU is needed; everything runs on an ordinary CPU. The first run downloads about 350 small files and takes roughly 15–30 minutes, depending on how fast FIRMS responds. Finished periods are cached in `data/raw/firms/`, so later runs take a minute or two.

**Read the numbers carefully.** A detection is one 375 m pixel that contained fire *at the moment the satellite passed*. It is not one field, and counts are not burnt area. Fires lit after the overpass, hidden by cloud or smoke, or too small are missed. Notebook 02 measures burnt area directly.

In [ ]:
# Colab: add two libraries its default image may lack. On your own computer, install requirements.txt
# into a virtual environment first; there this cell only checks that the install worked.
import importlib.util, subprocess, sys

try:
    import google.colab  # noqa: F401  (this module exists only on Colab)
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'geopandas', 'pyarrow'], check=True)
else:
    missing = [name for name in ('geopandas', 'pyarrow', 'matplotlib', 'requests')
               if importlib.util.find_spec(name) is None]
    if missing:
        raise ModuleNotFoundError('Missing: ' + ', '.join(missing) + '. In the VS Code terminal, activate the '
                                  'virtual environment, run  pip install -r requirements.txt  and re-run.')
    print('Libraries found.')

## 1. Settings

In [ ]:
import os, io, json, re, time, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib import patheffects
from matplotlib.patches import Patch
import geopandas as gpd
from IPython.display import display

TODAY = pd.Timestamp.today().normalize()
ACCESS_DATE = TODAY.strftime('%Y-%m-%d')

# On Colab, the project lives on Google Drive so downloads survive restarts.
try:
    from google.colab import drive, userdata  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/India-Space-Mission-Intelligence')
elif os.environ.get('PROJECT_DIR'):
    PROJECT = Path(os.environ['PROJECT_DIR']).resolve()
else:
    # On your computer: walk up from the working folder to the project root (the folder with requirements.txt).
    here = Path.cwd().resolve()
    PROJECT = next((p for p in (here, *here.parents)
                    if (p / 'requirements.txt').exists() and (p / 'notebooks').is_dir()), here.parent)

RAW = PROJECT / 'data' / 'raw'
PROCESSED = PROJECT / 'data' / 'processed'
FIGS = PROJECT / 'maps' / 'notebook01'
SOURCES = PROJECT / 'sources'
for folder in (RAW / 'firms', RAW / 'boundaries', PROCESSED, FIGS, SOURCES):
    folder.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 10})
print('Project folder:', PROJECT)

In [ ]:
# --- Study area -----------------------------------------------------------------
STATES = ['Punjab', 'Haryana']
BBOX = (73.8, 27.6, 77.6, 32.6)        # west, south, east, north in degrees: both states plus a margin

# --- Season window: paddy harvest and residue burning ----------------------------
SEASON_START = '09-15'                  # month-day
SEASON_END = '11-30'

# --- Which FIRMS product to download for which seasons ---------------------------
# SP  = standard processing (science quality, released with a lag)
# NRT = near real time (used for the current 2026 season)
DOWNLOAD_PLAN = [
    ('VIIRS_SNPP_SP',    2012, 2025),   # longest single-sensor record (FIRMS archive starts Jan 2012)
    ('VIIRS_NOAA20_SP',  2020, 2025),   # overlaps S-NPP; lets us convert counts once S-NPP stops
    ('VIIRS_SNPP_NRT',   2026, 2026),   # NOAA ends Suomi NPP data delivery on 2 Nov 2026
    ('VIIRS_NOAA20_NRT', 2026, 2026),
    ('VIIRS_NOAA21_NRT', 2026, 2026),
]
TREND_SOURCE = 'VIIRS_SNPP_SP'          # one sensor for every trend chart, so seasons compare fairly
DAY_RANGE = 5                           # days per request: the FIRMS API accepts 1-5 (its docs still say 1-10).
                                        # If FIRMS asks for fewer, the download lowers this automatically.
PAUSE_S = 1.0                           # pause after each live request, to be polite to the API
EXCLUDE_LOW_CONFIDENCE = False          # VIIRS confidence is l / n / h; True drops 'l'

# --- Charts and district shortlist ------------------------------------------------
STATE_COLOURS = {'Punjab': 'tab:orange', 'Haryana': 'tab:blue'}   # the same colour for a state in every chart
MIN_DETECTIONS_FOR_CHANGE = 100         # the change map greys out districts with fewer 2021 detections than this
SHORTLIST_N = 3                         # shortlist: top N by the 2021-2025 total plus top N by 2025 alone

# Districts whose geoBoundaries polygon is not today's district: (short label for charts, note for outputs).
BOUNDARY_NOTES = {
    ('Sangrur', 'Punjab'): ('pre-2021 boundary',
                            'Pre-2021 boundary: includes Malerkotla, carved out of Sangrur on 2 June 2021 '
                            '(https://malerkotla.nic.in/about-district/, accessed 2026-10-22)'),
}


def read_env_file(path):
    # KEY=value lines from a .env file; blank lines and lines starting with # are skipped.
    values = {}
    if path.exists():
        for line in path.read_text(encoding='utf-8-sig').splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                name, value = line.split('=', 1)
                values[name.strip()] = value.strip().strip('"\'')
    return values


def get_map_key():
    key = None
    if IN_COLAB:
        try:
            key = userdata.get('FIRMS_MAP_KEY')
        except Exception:
            key = None
    key = key or os.environ.get('FIRMS_MAP_KEY') or read_env_file(PROJECT / '.env').get('FIRMS_MAP_KEY')
    if not key:
        if IN_COLAB:
            hint = ('In Colab, click the key icon (Secrets) in the left sidebar, add a secret named FIRMS_MAP_KEY, '
                    'switch on notebook access, then re-run this cell.')
        else:
            hint = (f'On your computer, create a file named .env in {PROJECT} containing one line, '
                    'FIRMS_MAP_KEY=your_key, then re-run this cell.')
            if (PROJECT / '.env.txt').exists():
                hint += ' A file named .env.txt exists: rename it to .env.'
        raise RuntimeError('No FIRMS map key found. ' + hint)
    return key.strip()


MAP_KEY = get_map_key()
print('Map key loaded (not printed).')

## 2. Helper functions

All network calls go through `http_get`. Downloads are cached per block of up to five days, but a block that ends within the last two days is not cached, because near-real-time data for recent days can still change. Every dataset we touch is written to `sources/source_register.csv`, with the fields the brief's Section 8 asks for.

In [ ]:
FIRMS_API = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/{key}/{source}/{bbox}/{days}/{date}'
GB_URLS = ['https://www.geoboundaries.org/api/current/gbOpen/{iso}/{adm}/',
           'https://www.geoboundaries.org/gbRequest.html?ISO={iso}&ADM={adm}']
HDX_PAGE = 'https://data.humdata.org/m/dataset/geoboundaries-admin-boundaries-for-india'
REGISTER = SOURCES / 'source_register.csv'
REGISTER_COLUMNS = ['source_id', 'dataset_or_product', 'provider', 'url', 'date_accessed', 'geography',
                    'time_range', 'resolution', 'licence_or_terms', 'preprocessing', 'known_limitations', 'used_in']


def http_get(url, timeout=120):
    return requests.get(url, timeout=timeout)


def register_source(**row):
    # Add or replace one row of the source register, keyed by source_id.
    row = {c: str(row.get(c, '')) for c in REGISTER_COLUMNS}
    if REGISTER.exists():
        reg = pd.read_csv(REGISTER, dtype=str).fillna('')
    else:
        reg = pd.DataFrame(columns=REGISTER_COLUMNS)
    reg = reg[reg['source_id'] != row['source_id']]
    reg = pd.concat([reg, pd.DataFrame([row])], ignore_index=True)
    reg[REGISTER_COLUMNS].to_csv(REGISTER, index=False)


class DayRangeTooLong(Exception):
    # FIRMS refused the number of days in one request; args[0] is the most it accepts.
    pass


def fetch_chunk(source, start, days):
    # Download one block of `days` days of detections for the bounding box.
    start_ts = pd.Timestamp(start)
    end_ts = start_ts + pd.Timedelta(days - 1, unit='D')
    cache = RAW / 'firms' / source / f'{start_ts:%Y-%m-%d}_{days}d.csv'
    if cache.exists():
        return pd.read_csv(cache)
    url = FIRMS_API.format(key=MAP_KEY, source=source, bbox=','.join(map(str, BBOX)),
                           days=days, date=f'{start_ts:%Y-%m-%d}')
    waits = [60, 120, 300, 600]
    message = ''
    for attempt in range(len(waits) + 1):
        try:
            r = http_get(url)
            text = r.text.strip()
            if r.status_code == 200 and (text == '' or text.startswith('latitude')):
                df = pd.read_csv(io.StringIO(text)) if text else pd.DataFrame(columns=['latitude', 'longitude'])
                if end_ts < TODAY - pd.Timedelta(2, unit='D'):  # cache only periods that are complete
                    cache.parent.mkdir(parents=True, exist_ok=True)
                    df.to_csv(cache, index=False)
                time.sleep(PAUSE_S)
                return df
            message = text[:200]
        except requests.RequestException as err:
            message = str(err).replace(MAP_KEY, '***')[:200]  # requests puts the URL, key included, in its errors
        limit = re.search(r'day range.*?\[1\s*\.\.\s*(\d+)\]', message, flags=re.I)
        if limit:
            raise DayRangeTooLong(int(limit.group(1)))
        if 'invalid' in message.lower():
            hint = ' Check that FIRMS_MAP_KEY holds the key from your FIRMS email.' if 'key' in message.lower() else ''
            raise RuntimeError(f'FIRMS rejected the request: {message}.{hint}')
        if attempt < len(waits):
            print(f'  {source} {start_ts:%Y-%m-%d}: {message!r}; retrying in {waits[attempt]} s')
            time.sleep(waits[attempt])
    raise RuntimeError(f'Could not download {source} from {start_ts:%Y-%m-%d}: {message}')


def season_window(year):
    start = pd.Timestamp(f'{year}-{SEASON_START}')
    end = min(pd.Timestamp(f'{year}-{SEASON_END}'), TODAY)
    return start, end


def download_season(source, year):
    global DAY_RANGE
    start, end = season_window(year)
    frames, day = [], start
    while day <= end:
        days = int(min(DAY_RANGE, (end - day).days + 1))
        try:
            chunk = fetch_chunk(source, day, days)
        except DayRangeTooLong as err:
            accepted = err.args[0]
            if not 1 <= accepted < days:
                raise RuntimeError(f'FIRMS refused a {days}-day request but says it accepts up to {accepted} days.')
            DAY_RANGE = accepted
            print(f'  FIRMS accepts at most {accepted} days per request; continuing with {accepted}-day blocks.')
            continue
        if len(chunk):
            frames.append(chunk.assign(source=source))
        day += pd.Timedelta(days, unit='D')
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def get_boundaries(adm, iso='IND'):
    # Fetch geoBoundaries (gbOpen) for one admin level; cached in data/raw/boundaries.
    path = RAW / 'boundaries' / f'geoBoundaries-{iso}-{adm}.geojson'
    meta_path = RAW / 'boundaries' / f'geoBoundaries-{iso}-{adm}.meta.json'
    if not path.exists():
        meta = None
        for pattern in GB_URLS:
            try:
                answer = http_get(pattern.format(iso=iso, adm=adm), timeout=60).json()
                answer = answer[0] if isinstance(answer, list) else answer
                if answer.get('gjDownloadURL'):
                    meta = answer
                    break
            except Exception as err:
                print('  geoBoundaries lookup failed:', str(err)[:120])
        if meta is None:
            raise RuntimeError(f'Could not reach geoBoundaries. Download the India {adm} GeoJSON from {HDX_PAGE} '
                               f'and save it as {path}, then re-run this cell.')
        content = http_get(meta['gjDownloadURL'], timeout=600).content
        if not content.lstrip().startswith(b'{'):
            raise RuntimeError(f'The {adm} download was not GeoJSON. Download it from {HDX_PAGE} '
                               f'and save it as {path}, then re-run this cell.')
        path.write_bytes(content)
        meta_path.write_text(json.dumps({k: v for k, v in meta.items() if isinstance(v, (str, int, float))}, indent=2))
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    return gpd.read_file(path).to_crs(4326), meta


def plain_name(name):
    # Lower-case ASCII form of a place name, so geoBoundaries' 'Haryāna' matches 'Haryana'.
    return unicodedata.normalize('NFKD', str(name)).encode('ascii', 'ignore').decode('ascii').strip().lower()


def district_label(district, state, show_state=True):
    # Chart label for a district, flagged when its polygon is not today's district (see BOUNDARY_NOTES).
    parts = [state] if show_state else []
    if (district, state) in BOUNDARY_NOTES:
        parts.append(BOUNDARY_NOTES[(district, state)][0])
    return f'{district} ({", ".join(parts)})' if parts else district


def hhmm(hours):
    hours = float(hours)
    return f'{int(hours):02d}:{int((hours % 1) * 60):02d}'

## 3. Download fire detections

One request covers up to five days, so each season is about 16 requests per sensor. Suomi NPP (S-NPP) gives the long trend. NOAA-20 overlaps it from 2020, so Notebook 04 can convert counts once S-NPP stops. The 2026 season uses near-real-time data.

In [ ]:
frames = []
for source, first, last in DOWNLOAD_PLAN:
    for year in range(first, last + 1):
        season = download_season(source, year)
        print(f'{source:<18} {year}: {len(season):>8,} detections in the bounding box')
        if len(season):
            frames.append(season.assign(season=year))
if not frames:
    raise RuntimeError('Nothing downloaded. Check the map key, and the FIRMS status page if it persists.')
raw = pd.concat(frames, ignore_index=True)
print(f'\nDownloaded {len(raw):,} detections (before clipping to {" and ".join(STATES)}).')

for source, first, last in DOWNLOAD_PLAN:
    sensor = source.replace('VIIRS_', '').replace('_SP', '').replace('_NRT', '')
    product = 'standard processing (SP)' if source.endswith('_SP') else 'near real time (NRT)'
    register_source(
        source_id=f'firms_{source.lower()}',
        dataset_or_product=f'VIIRS 375 m active fire detections, {sensor}, {product}; FIRMS API source {source}',
        provider='NASA FIRMS (Fire Information for Resource Management System)',
        url='https://firms.modaps.eosdis.nasa.gov/api/area/',
        date_accessed=ACCESS_DATE,
        geography=f'Bounding box {BBOX}, clipped to {" and ".join(STATES)}',
        time_range=f'{SEASON_START} to {SEASON_END} each season, {first}-{last}',
        resolution='375 m fire pixels; detections only at satellite overpass times',
        licence_or_terms='NASA open data policy; cite per the FIRMS citation policy',
        preprocessing='Duplicates removed; vegetation-fire type kept where the field exists; UTC to IST; joined to districts',
        known_limitations='Points are pixel centres, not fire locations; fires between overpasses, under cloud or smoke, or too small are missed',
        used_in='notebooks/01_fire_counts.ipynb',
    )

## 4. District boundaries

In [ ]:
adm1, meta1 = get_boundaries('ADM1')
adm2, meta2 = get_boundaries('ADM2')

# geoBoundaries writes some names with diacritics ('Haryāna'), so match on plain names and keep the STATES spelling.
spelling = {plain_name(s): s for s in STATES}
states = adm1.assign(state=adm1['shapeName'].map(plain_name).map(spelling))
states = states.loc[states['state'].notna(), ['state', 'geometry']]
missing = set(STATES) - set(states['state'])
assert not missing, f'States not found in ADM1: {missing}. Available: {sorted(adm1["shapeName"])}'

if 'shapeID' not in adm2.columns:
    adm2['shapeID'] = adm2.index.astype(str)
centres = gpd.GeoDataFrame(adm2[['shapeID']].copy(), geometry=adm2.representative_point(), crs=4326)
in_states = gpd.sjoin(centres, states, predicate='within', how='inner')[['shapeID', 'state']]
districts = (adm2.merge(in_states, on='shapeID')
                 .rename(columns={'shapeName': 'district'})[['district', 'state', 'shapeID', 'geometry']])
print(districts.groupby('state').size().rename('districts'))

ax = districts.plot(color=districts['state'].map(STATE_COLOURS), edgecolor='white', linewidth=0.5, figsize=(6, 6))
ax.legend(handles=[Patch(color=colour, label=state) for state, colour in STATE_COLOURS.items()])
ax.set_title('Districts used in this analysis')
ax.set_axis_off()
plt.show()

boundary_notes = '; '.join(f'{d}, {s}: {note}' for (d, s), (_, note) in BOUNDARY_NOTES.items())
for adm, meta in (('ADM1', meta1), ('ADM2', meta2)):
    register_source(
        source_id=f'geoboundaries_ind_{adm.lower()}',
        dataset_or_product=f'geoBoundaries India {adm} (gbOpen)',
        provider='geoBoundaries (William & Mary geoLab)',
        url=meta.get('gjDownloadURL', HDX_PAGE),
        date_accessed=ACCESS_DATE,
        geography='India; Punjab and Haryana used',
        time_range=str(meta.get('boundaryYearRepresented', meta.get('boundaryYear', ''))),
        resolution='Vector polygons',
        licence_or_terms=meta.get('boundaryLicense', 'See the HDX page, which lists ODbL'),
        preprocessing='Districts assigned to a state by their representative point',
        known_limitations='Boundary vintage may predate newer districts; not an official Survey of India depiction'
                          + (f'. {boundary_notes}' if adm == 'ADM2' and boundary_notes else ''),
        used_in='notebooks/01_fire_counts.ipynb',
    )

## 5. Clean the detections and attach districts

Rules: drop exact duplicates; keep the "presumed vegetation fire" type where the archive provides that field (it removes known industrial heat sources); convert time to IST; keep only points inside the two states.

In [ ]:
df = raw.copy()
df['acq_time'] = pd.to_numeric(df['acq_time'], errors='coerce')
df = df.dropna(subset=['latitude', 'longitude', 'acq_date', 'acq_time'])
clock = df['acq_time'].astype(int).astype(str).str.zfill(4)
df['time_utc'] = pd.to_datetime(df['acq_date'].astype(str) + ' ' + clock, format='%Y-%m-%d %H%M',
                                utc=True, errors='coerce')
unreadable = df['time_utc'].isna().sum()
if unreadable:
    print(f'Skipping {unreadable:,} rows with an unreadable date or time.')
df = df.dropna(subset=['time_utc'])
df['time_ist'] = df['time_utc'].dt.tz_convert('Asia/Kolkata')
df['date_ist'] = df['time_ist'].dt.tz_localize(None).dt.normalize()
df['hour_ist'] = df['time_ist'].dt.hour + df['time_ist'].dt.minute / 60
df['sensor'] = df['source'].str.extract(r'VIIRS_(SNPP|NOAA20|NOAA21)', expand=False)
df['product'] = np.where(df['source'].str.endswith('_SP'), 'SP', 'NRT')

downloaded = len(df)
df = df.drop_duplicates(subset=['latitude', 'longitude', 'acq_date', 'acq_time', 'source'])
if 'type' in df.columns:
    df = df[df['type'].isna() | (df['type'] == 0)]
if EXCLUDE_LOW_CONFIDENCE:
    df = df[df['confidence'].astype(str).str.lower() != 'l']

points = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['longitude'], df['latitude']), crs=4326)
points = gpd.sjoin(points, districts[['district', 'state', 'geometry']], predicate='within', how='inner')
points = points.drop(columns='index_right')
print(f'{downloaded:,} downloaded -> {len(points):,} inside {" and ".join(STATES)} after cleaning')
display(points.groupby(['source', 'state']).size().unstack(fill_value=0))

keep = ['latitude', 'longitude', 'acq_date', 'acq_time', 'time_ist', 'hour_ist', 'date_ist', 'season', 'source',
        'sensor', 'product', 'confidence', 'frp', 'daynight', 'district', 'state']
pd.DataFrame(points[[c for c in keep if c in points.columns]]).to_parquet(
    PROCESSED / 'firms_viirs_punjab_haryana_points.parquet', index=False)

## 6. How have detections changed since 2012?

In [ ]:
trend_points = points[points['source'] == TREND_SOURCE]
trend = trend_points.groupby(['season', 'state']).size().unstack(fill_value=0)
trend = trend.reindex(columns=STATES, fill_value=0)
trend['Total'] = trend.sum(axis=1)
display(trend)
trend.to_csv(PROCESSED / 'firms_detections_by_season_state.csv')

fig, ax = plt.subplots(figsize=(8, 4))
for state in STATES:
    ax.plot(trend.index, trend[state], marker='o', label=state, color=STATE_COLOURS[state])
ax.set_title(f'VIIRS fire detections per paddy season ({SEASON_START} to {SEASON_END}), Suomi NPP')
ax.set_xlabel('Season')
ax.set_ylabel('Detections')
ax.legend()
fig.tight_layout()
fig.savefig(FIGS / '01_detections_by_season.png', dpi=200)
plt.show()

peak, last = trend['Total'].idxmax(), trend.index.max()
print(f'Peak season: {peak} with {trend.loc[peak, "Total"]:,} detections.')
print(f'{last}: {trend.loc[last, "Total"]:,} detections ({(trend.loc[last, "Total"] / trend.loc[peak, "Total"] - 1) * 100:+.0f}% vs peak).')
if 2021 in trend.index:
    print(f'2021 to {last}: {(trend.loc[last, "Total"] / trend.loc[2021, "Total"] - 1) * 100:+.0f}%. '
          'Remember: detections are not burnt area (Notebook 02 measures that).')

## 7. When in the day do satellites see fires?

Polar-orbiting satellites pass over a place at roughly the same local times every day. This chart shows the detection clock for the two VIIRS sensors.

In [ ]:
sp = points[(points['product'] == 'SP') & points['season'].between(2020, 2025)]
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.arange(0, 24.25, 0.25)
for sensor, label in (('SNPP', 'Suomi NPP'), ('NOAA20', 'NOAA-20')):
    hours = sp.loc[sp['sensor'] == sensor, 'hour_ist']
    if len(hours):
        ax.hist(hours, bins=bins, alpha=0.6, label=f'{label} ({len(hours):,} detections)')
ax.axvspan(15, 24, color='grey', alpha=0.08)
ax.text(15.3, ax.get_ylim()[1] * 0.92, 'after 3 pm', color='dimgrey')
ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 3))
ax.set_xlabel('Time of detection, IST (hour of day)')
ax.set_ylabel('Detections, 2020-2025 seasons')
ax.set_title('Satellites only see a fire when they pass overhead')
ax.legend()
fig.tight_layout()
fig.savefig(FIGS / '02_detection_time_ist.png', dpi=200)
plt.show()

for sensor in ('SNPP', 'NOAA20'):
    day = sp[(sp['sensor'] == sensor) & (sp['daynight'] == 'D')]['hour_ist'] if 'daynight' in sp else sp['hour_ist']
    if len(day):
        print(f'{sensor}: 90% of daytime detections between {hhmm(day.quantile(0.05))} and '
              f'{hhmm(day.quantile(0.95))} IST; after 3 pm: {(day >= 15).mean() * 100:.1f}%')

## 8. Season shape: when does burning peak?

In [ ]:
daily = trend_points.groupby(['season', 'date_ist']).size().rename('detections').reset_index()
daily['day'] = (daily['date_ist'] - pd.to_datetime(daily['season'].astype(str) + '-' + SEASON_START)).dt.days
season_days = (pd.Timestamp(f'2001-{SEASON_END}') - pd.Timestamp(f'2001-{SEASON_START}')).days

fig, ax = plt.subplots(figsize=(8, 4))
for year in [y for y in (2016, 2021, 2023, 2025) if y in set(daily['season'])]:
    s = daily[daily['season'] == year].set_index('day')['detections']
    s = s.reindex(range(season_days + 1), fill_value=0).rolling(7, center=True, min_periods=1).mean()
    ax.plot(s.index, s.values, label=str(year))
ticks = range(0, season_days + 1, 15)
ax.set_xticks(list(ticks))
ax.set_xticklabels([(pd.Timestamp(f'2001-{SEASON_START}') + pd.Timedelta(d, unit='D')).strftime('%d %b') for d in ticks])
ax.set_ylabel('Detections per day (7-day average)')
ax.set_title('Season shape, Suomi NPP')
ax.legend()
fig.tight_layout()
fig.savefig(FIGS / '03_season_shape.png', dpi=200)
plt.show()

peaks = daily.loc[daily.groupby('season')['detections'].idxmax(), ['season', 'date_ist', 'detections']]
display(peaks.rename(columns={'date_ist': 'peak_day', 'detections': 'peak_detections'}).set_index('season'))

## 9. Districts: where are fires detected most?

In [ ]:
recent = [str(y) for y in range(2021, 2026)]
by_district = trend_points.groupby(['district', 'state', 'season']).size().unstack(fill_value=0)
by_district.columns = [str(c) for c in by_district.columns]
for y in recent:
    if y not in by_district.columns:
        by_district[y] = 0
by_district['total_2021_2025'] = by_district[recent].sum(axis=1)
by_district['change_2021_2025_pct'] = np.where(by_district['2021'] > 0,
                                               (by_district['2025'] / by_district['2021'].where(by_district['2021'] > 0) - 1) * 100,
                                               np.nan)
by_district['boundary_note'] = [BOUNDARY_NOTES.get(key, ('', ''))[1] for key in by_district.index]
by_district = by_district.sort_values('total_2021_2025', ascending=False)
by_district.to_csv(PROCESSED / 'firms_detections_by_district_season.csv')
display(by_district[['2021', '2023', '2025', 'total_2021_2025', 'change_2021_2025_pct']].head(10).round(0))

top = by_district.head(10).reset_index().iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh([district_label(d, s) for d, s in zip(top['district'], top['state'])], top['total_2021_2025'],
        color=top['state'].map(STATE_COLOURS))
ax.set_xlabel('VIIRS detections, 2021-2025 seasons (Suomi NPP)')
ax.set_title('Top 10 districts by detections')
fig.tight_layout()
fig.savefig(FIGS / '04_top_districts.png', dpi=200)
plt.show()

m = districts.merge(by_district.reset_index()[['district', 'state', '2021', '2025', 'change_2021_2025_pct']],
                    on=['district', 'state'], how='left')
# A percentage change on a handful of detections is noise, so grey out districts with too few in 2021.
m['change_shown'] = m['change_2021_2025_pct'].where(m['2021'] >= MIN_DETECTIONS_FOR_CHANGE)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
m.plot(column='2025', cmap='OrRd', legend=True, ax=axes[0], edgecolor='white', linewidth=0.4,
       missing_kwds={'color': 'lightgrey'}, legend_kwds={'shrink': 0.6, 'label': 'Detections'})
axes[0].set_title('Detections in the 2025 season')
m.plot(column='change_shown', cmap='RdBu_r', vmin=-100, vmax=100, legend=True, ax=axes[1],
       edgecolor='white', linewidth=0.4, missing_kwds={'color': 'lightgrey'},
       legend_kwds={'shrink': 0.6, 'label': 'Change 2021 to 2025 (%)'})
axes[1].set_title(f'Change in detections, 2021 to 2025\n(grey: fewer than {MIN_DETECTIONS_FOR_CHANGE} detections in 2021)')
for _, row in m.sort_values('2025', ascending=False).head(5).iterrows():
    pt = row.geometry.representative_point()
    axes[0].annotate(district_label(row['district'], row['state'], show_state=False).replace(' (', '\n('),
                     (pt.x, pt.y), fontsize=8, ha='center',
                     path_effects=[patheffects.withStroke(linewidth=2.5, foreground='white')])
for a in axes:
    a.set_axis_off()
fig.suptitle('VIIRS fire detections by district, Punjab and Haryana (Suomi NPP)')
fig.tight_layout()
fig.savefig(FIGS / '05_district_maps.png', dpi=200)
plt.show()

## 10. Sensor ratio for the forecast (Notebook 04)

Suomi NPP data stops on 2 Nov 2026. The ratio of Suomi NPP to NOAA-20 detections in the seasons both sensors covered lets Notebook 04 express 2026 NOAA-20 counts in Suomi NPP terms.

In [ ]:
both = points[(points['product'] == 'SP') & points['season'].between(2020, 2025)]
ratio = both.groupby(['season', 'sensor']).size().unstack(fill_value=0)
if {'SNPP', 'NOAA20'} <= set(ratio.columns):
    ratio['snpp_per_noaa20'] = ratio['SNPP'] / ratio['NOAA20'].replace(0, np.nan)
    display(ratio.round(2))
    print(f"Mean ratio 2020-2025: {ratio['snpp_per_noaa20'].mean():.2f} "
          f"(range {ratio['snpp_per_noaa20'].min():.2f} to {ratio['snpp_per_noaa20'].max():.2f})")
    ratio.to_csv(PROCESSED / 'firms_snpp_noaa20_ratio.csv')
else:
    print('Need both SNPP and NOAA20 standard-processing data to compute the ratio.')

## 11. The 2026 season so far (near real time)

In [ ]:
live = points[points['season'] == 2026]
if len(live):
    to_date = live['date_ist'].max()
    print(f'2026 detections from {SEASON_START} to {to_date:%d %b %Y}, by sensor:')
    display(live.groupby('sensor').size().rename('detections').to_frame())
    same = trend_points[(trend_points['season'] == 2025) &
                        (trend_points['date_ist'] <= pd.Timestamp(f'2025-{to_date:%m-%d}'))]
    snpp_2026 = (live['sensor'] == 'SNPP').sum()
    print(f'Same dates in 2025 (Suomi NPP, standard processing): {len(same):,}. '
          f'2026 Suomi NPP (near real time): {snpp_2026:,}. NRT and SP differ slightly, so treat this as indicative.')
else:
    print('No 2026 detections in the window yet.')

## 12. District shortlist (Checkpoint 1)

Two rankings, both from Suomi NPP: the 2021–2025 total (the roadmap's default rule) and the 2025 season alone, which is the season the core question asks about. The shortlist holds the top `SHORTLIST_N` districts by each ranking.

**Decision (22 Oct 2026, recorded in `decision_log.md`):** Sangrur, on its pre-2021 boundary (which includes Malerkotla), with Tarn Taran as the fallback. Sentinel-2 coverage is checked in Notebook 02.

In [ ]:
ranked = by_district.reset_index()
ranked['rank_2021_2025'] = ranked['total_2021_2025'].rank(ascending=False, method='min').astype(int)
ranked['rank_2025'] = ranked['2025'].rank(ascending=False, method='min').astype(int)
ranked['share_of_both_states_pct'] = ranked['total_2021_2025'] / ranked['total_2021_2025'].sum() * 100
ranked['share_of_both_states_2025_pct'] = ranked['2025'] / ranked['2025'].sum() * 100
shortlist = ranked[(ranked['rank_2021_2025'] <= SHORTLIST_N) | (ranked['rank_2025'] <= SHORTLIST_N)]
shortlist = shortlist[['district', 'state', '2021', '2025', 'total_2021_2025', 'change_2021_2025_pct',
                       'share_of_both_states_pct', 'share_of_both_states_2025_pct', 'rank_2021_2025', 'rank_2025',
                       'boundary_note']]
display(shortlist.drop(columns='boundary_note').round(1))
for _, row in shortlist[shortlist['boundary_note'] != ''].iterrows():
    print(f"{row['district']}: {row['boundary_note']}")
shortlist.to_csv(PROCESSED / 'district_shortlist.csv', index=False)
print('Done. Send Claude this table, the five charts in maps/notebook01/ and any warnings above.')

### Caveats to carry into the report
- **Counts are not area.** One detection is a 375 m pixel with fire at overpass time; Notebook 02 maps burnt area from Sentinel-2.
- **Timing bias.** VIIRS passes in the early afternoon; fires lit later are missed (see section 7).
- **Sensor changes.** Trend charts use one sensor (Suomi NPP). NOAA-20 and NOAA-21 detect more fires in total because there are more overpasses; never add them into a trend.
- **Boundaries.** geoBoundaries districts may predate newer districts; the source register records the licence. Sangrur's polygon is the pre-2021 district: it includes Malerkotla, carved out of Sangrur on 2 June 2021 ([Malerkotla district website](https://malerkotla.nic.in/about-district/), accessed 22 Oct 2026). Outputs label it "pre-2021 boundary".
- **Small counts.** The change map greys out districts with fewer than 100 detections in 2021, where a percentage change would rest on a handful of fires.